# Integration of prefix-level ecnoded data into the pipeline

Question: can models (log reg / rand forest) run on this data?

First prepare the data accroding to the pipeline

In [1]:
from pathlib import Path
import sys

project_root = Path().resolve().parent
sys.path.append(str(project_root))

In [2]:
from src.featureEngineering.feature_encoding import Encode
from src.dataExtraction.extract import split, import_data
from src.featureEngineering.outcome_labelling import outcome

# Import data

data_path = project_root / "data" / "raw" / "BPI_Challenge_2013_incidents" / "BPI_Challenge_2013_incidents.xes"
drop_columns = ["impact", "org:role"]
df = import_data(str(data_path), drop_columns=drop_columns)


/Users/so/Desktop/explainable-process-prediction/.venv/lib/python3.13/site-packages/pm4py/utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(
/Users/so/Desktop/explainable-process-prediction/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
parsing log, completed traces :: 100%|██████████| 7554/7554 [00:01<00:00, 7069.34it/s]


      org:group resource country organization country org:resource  \
0           V30           France                   fr     Frederic   
1           V30           France                   fr     Frederic   
2        V5 3rd           France                   fr     Frederic   
3        V5 3rd           France                   fr  Anne Claire   
4           V30           France                   fr  Anne Claire   
...         ...              ...                  ...          ...   
65528        C9           Brazil                   br      Lierson   
65529        C9           Brazil                   br      Lierson   
65530        C9           Brazil                   br      Lierson   
65531        C9           Brazil                   br      Lierson   
65532       N36              USA                   us         Matt   

      organization involved concept:name  product lifecycle:transition  \
0               Org line A2     Accepted  PROD582          In Progress   
1          

In [3]:
df = df.copy()

case_duration = (
    df.groupby("case:concept:name")["time:timestamp"]
    .agg(lambda x: (x.max() - x.min()).total_seconds())
)

df["case_duration_seconds"] = df["case:concept:name"].map(case_duration)

duration_threshold = (
    df.groupby("case:concept:name")["case_duration_seconds"]
    .first()
    .median()
)

outcome_rules = [
    {
        "feature": "case_duration_seconds",
        "operator": "always_le",
        "value": duration_threshold,
    }
]

df = outcome(df, outcome_rules)

print(df.groupby("case:concept:name")["outcome"].first().value_counts())

outcome
False    3777
True     3777
Name: count, dtype: int64


Now split the data and generate prefixes for each part.

In [4]:
train_df, val_df, test_df = split(df)

print(train_df["case:concept:name"].nunique())
print(val_df["case:concept:name"].nunique())
print(test_df["case:concept:name"].nunique())

5287
1133
1134


In [5]:
from src.featureEngineering.prefix_generation import generate_prefix

train_prefix_df = generate_prefix(train_df)
val_prefix_df = generate_prefix(val_df)
test_prefix_df = generate_prefix(test_df)

print(train_prefix_df["case:concept:name"].nunique())
print(val_prefix_df["case:concept:name"].nunique())
print(test_prefix_df["case:concept:name"].nunique())

46644
5894
5441


The number of cases has become much larger since for each initial case several prefixes were generated. Now check the labels.

In [6]:
print(train_prefix_df.groupby("case:concept:name")["outcome"].first().value_counts())
print(val_prefix_df.groupby("case:concept:name")["outcome"].first().value_counts())
print(test_prefix_df.groupby("case:concept:name")["outcome"].first().value_counts())

outcome
False    36735
True      9909
Name: count, dtype: int64
outcome
True     3704
False    2190
Name: count, dtype: int64
outcome
True     3511
False    1930
Name: count, dtype: int64


The next step would be to encode prefix data with aligned columns.

In [7]:
X_train_prefix, y_train_prefix, case_ids_train_prefix, prefix_feature_columns = Encode(train_prefix_df)

X_val_prefix, y_val_prefix, case_ids_val_prefix, _ = Encode(
    val_prefix_df,
    feature_columns=prefix_feature_columns,
)

X_test_prefix, y_test_prefix, case_ids_test_prefix, _ = Encode(
    test_prefix_df,
    feature_columns=prefix_feature_columns,
)

print(X_train_prefix.shape)
print(X_val_prefix.shape)
print(X_test_prefix.shape)

print(len(prefix_feature_columns))

(46644, 153)
(5894, 153)
(5441, 153)
153


## Prefix-related leakage

- case_duration -> direct leakage, since it represents final duration
- relative_age -> may cause indirect leakage since it can directly encode duration normalization

- elapsed_time and time_since_last -> can stay since they describe the prefix so far and don't reveal information from the whole case


In [8]:
prefix_leakage_features = [
    col for col in prefix_feature_columns
    if any(
        keyword in col.lower()
        for keyword in [
            "case_duration",
            "relative_age",
        ]
    )
]

print(prefix_leakage_features)
print(len(prefix_leakage_features))

['case_duration_seconds_sum', 'case_duration_seconds_mean', 'case_duration_seconds_min', 'case_duration_seconds_max', 'case_duration_seconds_std', 'case_duration_seconds_count', 'relative_age_sum', 'relative_age_mean', 'relative_age_min', 'relative_age_max', 'relative_age_std', 'relative_age_count']
12


In [9]:
prefix_keep_feature_indices = [
    i for i, col in enumerate(prefix_feature_columns)
    if col not in prefix_leakage_features
]

prefix_feature_columns_no_leakage = [
    prefix_feature_columns[i] for i in prefix_keep_feature_indices
]

X_train_prefix_no_leakage = X_train_prefix[:, prefix_keep_feature_indices]
X_val_prefix_no_leakage = X_val_prefix[:, prefix_keep_feature_indices]
X_test_prefix_no_leakage = X_test_prefix[:, prefix_keep_feature_indices]

print(X_train_prefix_no_leakage.shape)
print(X_val_prefix_no_leakage.shape)
print(X_test_prefix_no_leakage.shape)

(46644, 141)
(5894, 141)
(5441, 141)


## Train log reg on prefix data

In [10]:
from src.modeling.models import train_logistic_regression
from src.modeling.evaluation import evaluate_model

logreg_prefix = train_logistic_regression(
    X_train_prefix_no_leakage,
    y_train_prefix
)

logreg_prefix_result = evaluate_model(
    logreg_prefix,
    X_val_prefix_no_leakage,
    y_val_prefix,
    "Logistic Regression prefix",
)

logreg_prefix_result

/Users/so/Desktop/explainable-process-prediction/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


{'accuracy': 0.6479470648116729,
 'f1': 0.7635866469180813,
 'precision': 0.6605558840922531,
 'recall': 0.9046976241900648,
 'roc_auc': 0.6871253587384243,
 'model': 'Logistic Regression prefix'}

Evaluation results tell us, that the model predicts many cases as possible, which causes high recall, but the roc-auc results are only moderate. This can be explained by prefix prediction being harder than full-case prediction.

## Random Forest for prefix-level predictions

We use a smaller grid first since the prefix data is already much bigger.

In [11]:
from src.modeling.model_selection import select_best_random_forest

prefix_param_grid = [
    {"n_estimators": 100, "max_depth": 5},
    {"n_estimators": 100, "max_depth": 10},
    {"n_estimators": 200, "max_depth": 5},
]

best_rf_prefix, best_params_prefix, validation_table_prefix = select_best_random_forest(
    X_train_prefix_no_leakage,
    y_train_prefix,
    X_val_prefix_no_leakage,
    y_val_prefix,
    scoring="f1",
    param_grid=prefix_param_grid,
)

print(best_params_prefix)
validation_table_prefix

{'n_estimators': 100, 'max_depth': 10}


,accuracy,f1,precision,recall,roc_auc,model,params
0,0.638616,0.770028,0.641598,0.962743,0.729630,"Random Forest {'n_estimators': 100, 'max_depth...","{'n_estimators': 100, 'max_depth': 5}"
1,0.700882,0.795214,0.697859,0.924136,0.729420,"Random Forest {'n_estimators': 100, 'max_depth...","{'n_estimators': 100, 'max_depth': 10}"
2,0.638616,0.770127,0.641496,0.963283,0.729902,"Random Forest {'n_estimators': 200, 'max_depth...","{'n_estimators': 200, 'max_depth': 5}"


## Model evaluation and comparison

In [12]:
from src.modeling.evaluation import build_evaluation_table

best_rf_prefix_val_result = validation_table_prefix.loc[
    validation_table_prefix["params"].astype(str) == str(best_params_prefix)
].iloc[0].to_dict()

prefix_validation_comparison = build_evaluation_table([
    logreg_prefix_result,
    best_rf_prefix_val_result,
])

prefix_validation_comparison

,model,accuracy,precision,recall,f1,roc_auc
0,Logistic Regression prefix,0.647947,0.660556,0.904698,0.763587,0.687125
1,"Random Forest {'n_estimators': 100, 'max_depth...",0.700882,0.697859,0.924136,0.795214,0.729420


We can see that random forest performs stabely better on validation data.

Now we can check the performance on the test data.

In [13]:
from src.modeling.evaluation import evaluate_model, build_evaluation_table

prefix_test_results = [
    evaluate_model(
        logreg_prefix,
        X_test_prefix_no_leakage,
        y_test_prefix,
        "Logistic Regression prefix",
    ),
    evaluate_model(
        best_rf_prefix,
        X_test_prefix_no_leakage,
        y_test_prefix,
        "Random Forest prefix",
    ),
]

prefix_test_table = build_evaluation_table(prefix_test_results)
prefix_test_table

,model,accuracy,precision,recall,f1,roc_auc
0,Logistic Regression prefix,0.666789,0.686348,0.890629,0.775257,0.701037
1,Random Forest prefix,0.680941,0.745098,0.768442,0.756590,0.725977
